# Temperature Climate Projections

**Methodology**: Image-based approach using Earth Engine

**Standards:**
- Baseline: 1995-2014 (IPCC AR6)
- Scenarios: SSP2-4.5 (RCP4.5), SSP5-8.5 (RCP8.5)
- Windows: 2030s (2021–2040) 2050s (2041-2060), 2100s (2081-2100)
- Resolution: ERAN5 Land 0.1° (~9 km)
- Units: k -> C

In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import geemap
import ee, datetime

In [21]:
# Access Earth Engine
ee.Authenticate()
ee.Initialize(
    project = 'eecc-maureen',
    opt_url = 'https://earthengine-highvolume.googleapis.com'
)

## Region

In [3]:
# Place name
place_name = "Porto Alegre, Brazil"

# Get place boundary
roi = ox.geocode_to_gdf(place_name)
roi = gpd.GeoDataFrame(roi, geometry='geometry', crs='EPSG:4326')

# Convert geometry to Earth Engine format - Handle both Polygon and MultiPolygon
geom = roi.geometry.iloc[0]

if geom.geom_type == 'Polygon':
    coords = [list(geom.exterior.coords)]
    roi_ee = ee.Geometry.Polygon(coords)
elif geom.geom_type == 'MultiPolygon':
    polygons = []
    for poly in geom.geoms:
        polygons.append(list(poly.exterior.coords))
    roi_ee = ee.Geometry.MultiPolygon(polygons)
else:
    raise ValueError(f"Unexpected geometry type: {geom.geom_type}")

print(f"Geometry type: {geom.geom_type}")
print(f"Earth Engine geometry created: {roi_ee.getInfo()['type']}")

Geometry type: MultiPolygon
Earth Engine geometry created: MultiPolygon


## Collections & Constants

In [9]:
# Static constants (can be defined here)
BASE_START, BASE_END = '1995-01-01', '2015-01-01'

WINS = {
    '2030s': ('2021-01-01', '2041-01-01'), #end year is 2040 -> not inclusive   
    '2050s': ('2041-01-01', '2061-01-01'), #end year is 2060 
    '2100s': ('2081-01-01', '2101-01-01') #end year is 2100
}

SCENARIOS = {
    'RCP4.5': 'ssp245',
    'RCP8.5': 'ssp585'
}

MODELS = [
    'MIROC6', #'CESM2', 
    'MRI-ESM2-0', 'ACCESS-ESM1-5', 'EC-EARTH3'
]

## Helpers

In [10]:
cmip6 = ee.ImageCollection('NASA/GDDP-CMIP6')

def get_nex_tasmax_historical(model):
    """
    Historical daily tasmax in °C (scenario = 'historical').
    """
    ic = (cmip6
          .filterBounds(roi_ee)
          .filter(ee.Filter.eq('model', model))
          .filter(ee.Filter.eq('scenario', 'historical'))
          .select('tasmax'))
    
    def to_celsius(img):
        tx = img.subtract(273.15).rename('tasmax')  # K -> °C
        return tx.copyProperties(img, img.propertyNames())
    
    return ic.map(to_celsius)


def get_nex_tasmin_historical(model):
    """
    Historical daily tasmin in °C (scenario = 'historical').
    """
    ic = (cmip6
          .filterBounds(roi_ee)
          .filter(ee.Filter.eq('model', model))
          .filter(ee.Filter.eq('scenario', 'historical'))
          .select('tasmin'))
    
    def to_celsius(img):
        tn = img.subtract(273.15).rename('tasmin')
        return tn.copyProperties(img, img.propertyNames())
    
    return ic.map(to_celsius)


def get_nex_tasmax_scenario(model, scenario_code):
    """
    Future daily tasmax in °C for a given SSP (e.g. 'ssp245', 'ssp585').
    """
    ic = (cmip6
          .filterBounds(roi_ee)
          .filter(ee.Filter.eq('model', model))
          .filter(ee.Filter.eq('scenario', scenario_code))
          .select('tasmax'))
    
    def to_celsius(img):
        tx = img.subtract(273.15).rename('tasmax')
        return tx.copyProperties(img, img.propertyNames())
    
    return ic.map(to_celsius)


def get_nex_tasmin_scenario(model, scenario_code):
    """
    Future daily tasmin in °C for a given SSP.
    """
    ic = (cmip6
          .filterBounds(roi_ee)
          .filter(ee.Filter.eq('model', model))
          .filter(ee.Filter.eq('scenario', scenario_code))
          .select('tasmin'))
    
    def to_celsius(img):
        tn = img.subtract(273.15).rename('tasmin')
        return tn.copyProperties(img, img.propertyNames())
    
    return ic.map(to_celsius)


In [11]:
# Baseline TX percentiles (p90, p99) per model, per pixel
def compute_tx_baseline_percentiles_image(model):
    """
    Compute 90th and 99th percentiles of daily max temperature (tasmax, °C)
    over the baseline (BASE_START–BASE_END) for ONE model (historical).
    Returns (p90_img, p99_img) as ee.Image, per-pixel.
    """
    tx_ic = get_nex_tasmax_historical(model)
    base_ic = tx_ic.filterDate(BASE_START, BASE_END)

    percs = base_ic.reduce(ee.Reducer.percentile([90, 99]))
    p90 = percs.select('tasmax_p90').rename('TX_p90')
    p99 = percs.select('tasmax_p99').rename('TX_p99')

    return p90, p99


In [12]:
def compute_tx_indices_for_period(model, scenario_code, start, end,
                                  tx_p90_img, tx_p99_img):
    """
    Compute TXx, TNx, TX90p_days, TX99p_days as gridded ee.Image
    for a given model + scenario + period.
    All temperatures in °C.
    """
    # Daily temps for this model+scenario+period
    tx_ic = get_nex_tasmax_scenario(model, scenario_code).filterDate(start, end)
    tn_ic = get_nex_tasmin_scenario(model, scenario_code).filterDate(start, end)

    # --- TXx: hottest day (max daily max temp) ---
    TXx = tx_ic.max().rename('TXx')

    # --- TNx: hottest night (max daily min temp) ---
    TNx = tn_ic.max().rename('TNx')

    # --- TX90p_days: number of days where TX > baseline 90th percentile ---
    def flag_tx90(img):
        # img: tasmax (°C)
        return img.gt(tx_p90_img).rename('TX90p_flag')

    tx90_flags = tx_ic.map(flag_tx90)
    TX90p_days = tx90_flags.sum().rename('TX90p_days')

    # --- TX99p_days: number of days where TX > baseline 99th percentile ---
    def flag_tx99(img):
        return img.gt(tx_p99_img).rename('TX99p_flag')

    tx99_flags = tx_ic.map(flag_tx99)
    TX99p_days = tx99_flags.sum().rename('TX99p_days')

    # Pack all indices as one multi-band image, cast to float, clip to ROI
    indices_img = ee.Image.cat([TXx, TNx, TX90p_days, TX99p_days]) \
        .toFloat() \
        .clip(roi_ee) \
        .set('model', model) \
        .set('scenario', scenario_code) \
        .set('start', start) \
        .set('end', end)

    return indices_img

## Calculations

### Period 2030 - Scenario ssp245

In [13]:
period_name = '2030s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP4.5']  # 'ssp245'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()

Skipping EC-EARTH3 for ssp245 2030s: only 0 days
Temperature models used: 3


In [14]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Period 2030 - Scenario ssp585

In [15]:
period_name = '2030s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP8.5']  # 'ssp285'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()


Skipping EC-EARTH3 for ssp585 2030s: only 0 days
Temperature models used: 3


In [16]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Period 2050 - Scenario ssp245

In [17]:
period_name = '2050s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP4.5']  # 'ssp245'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()

Skipping EC-EARTH3 for ssp245 2050s: only 0 days
Temperature models used: 3


In [18]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Period 2050 - Scenario ssp585

In [22]:
period_name = '2050s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP8.5']  # 'ssp285'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()

Skipping EC-EARTH3 for ssp585 2050s: only 0 days
Temperature models used: 3


In [23]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Period 2100 - Scenario ssp245

In [24]:
period_name = '2100s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP4.5']  # 'ssp245'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()

Skipping EC-EARTH3 for ssp245 2100s: only 0 days
Temperature models used: 3


In [25]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()

### Period 2100 - Scenario ssp585

In [26]:
period_name = '2100s'
start, end = WINS[period_name]
scenario_code = SCENARIOS['RCP8.5']  # 'ssp285'

# 1) precompute baseline TX p90/p99 per model (historical)
tx_p90p99_by_model = {
    m: compute_tx_baseline_percentiles_image(m)  # (p90_img, p99_img)
    for m in MODELS
}

tx_indices_images = []

for m in MODELS:
    # optional: check availability
    tx_ic_test = get_nex_tasmax_scenario(m, scenario_code).filterDate(start, end)
    n = tx_ic_test.size().getInfo()
    if n < 10:  # at least ~10 days to be safe
        print(f"Skipping {m} for {scenario_code} {period_name}: only {n} days")
        continue

    tx_p90_img, tx_p99_img = tx_p90p99_by_model[m]

    idx_img = compute_tx_indices_for_period(
        model=m,
        scenario_code=scenario_code,
        start=start,
        end=end,
        tx_p90_img=tx_p90_img,
        tx_p99_img=tx_p99_img,
    )
    tx_indices_images.append(idx_img)

print("Temperature models used:", len(tx_indices_images))
tx_ic = ee.ImageCollection(tx_indices_images)

# Ensemble mean (still grid)
tx_ensemble_mean = tx_ic.mean().rename(
    ['TXx_mean', 'TNx_mean', 'TX90p_days_mean', 'TX99p_days_mean']
).toFloat()

Skipping EC-EARTH3 for ssp585 2100s: only 0 days
Temperature models used: 3


In [27]:
task = ee.batch.Export.image.toDrive(
    image=tx_ensemble_mean,
    description="Extreme Temperature Indices Projections",
    folder="GEE_exports",
    fileNamePrefix=f"Extreme_Temperature_Indices_Projections_{period_name}_{scenario_code}",
    region=roi_ee,
    scale=11132,          
    crs="EPSG:4326",
    maxPixels=1e13
)
task.start()